# Eaton Fire Corridor Analysis
**I-210 EB/WB and SR-134 WB — January 7, 2025**

Compares fire day traffic (Jan 7, 2025) against the averaged Tuesday baseline (Dec 3, 10, 17, 2024).

Fire ignition: **~18:18 PST** at Altadena/Eaton Canyon area.

---
- **Macro plots** (Cells 2–3): corridor-level aggregated flow and speed
- **Micro plots** (Cells 4–5): per-station breakdown colored by milepost

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
import os

# ── Paths ──────────────────────────────────────────────────────────────────
NB_DIR = os.path.abspath(".")   # run from data-eaton/analysis/
BASE = os.path.normpath(os.path.join(NB_DIR, "..", "pems", "eaton-corridor"))

FIRE_DIR     = os.path.join(BASE, "7thData-FireDay")
BASELINE_DIR = os.path.join(BASE, "BaselineData_Dec03-10-17")

SEGMENTS = {
    "I-210 EB":  ("eaton-i210", "eaton_i210_E"),
    "I-210 WB":  ("eaton-i210", "eaton_i210_W"),
    "SR-134 WB": ("eaton-134",  "eaton_134_W"),
}

FIRE_DATE     = "2025_01_07"
BASELINE_TAG  = "baseline_Dec03-10-17"
FIRE_IGNITION = "18:18:00"

# ── Loaders ────────────────────────────────────────────────────────────────
def load_ml_hv(folder, subfolder, prefix, tag):
    path = os.path.join(folder, subfolder, f"{prefix}_{tag}_ML_HV.csv")
    df = pd.read_csv(path, low_memory=False)
    ml = df[df["Lane Type"] == "ML"].copy()
    hv = df[df["Lane Type"] == "HV"].copy()
    return ml, hv

fire_ml,     fire_hv     = {}, {}
baseline_ml, baseline_hv = {}, {}

for lbl, (sub, pre) in SEGMENTS.items():
    fire_ml[lbl],     fire_hv[lbl]     = load_ml_hv(FIRE_DIR,     sub, pre, FIRE_DATE)
    baseline_ml[lbl], baseline_hv[lbl] = load_ml_hv(BASELINE_DIR, sub, pre, BASELINE_TAG)

# Keep legacy names pointing to ML-only for macro plots
fire_data     = fire_ml
baseline_data = baseline_ml

# ── Normalise timestamps to HH:MM:SS ──────────────────────────────────────
for lbl in SEGMENTS:
    for d in [fire_ml, fire_hv]:
        d[lbl]["Time"] = pd.to_datetime(d[lbl]["Timestamp"]).dt.strftime("%H:%M:%S")
    for d in [baseline_ml, baseline_hv]:
        d[lbl]["Time"] = d[lbl]["Timestamp"]   # already HH:MM:SS

# ── Corridor-level aggregation (sum flow, mean speed) ─────────────────────
def corridor_agg(df):
    return (df.groupby("Time")
              .agg(Total_Flow=("Total Flow (veh/5min)", "sum"),
                   Avg_Speed  =("Avg Speed (mph)",      "mean"))
              .reset_index().sort_values("Time"))

fire_agg     = {lbl: corridor_agg(v) for lbl, v in fire_ml.items()}
baseline_agg = {lbl: corridor_agg(v) for lbl, v in baseline_ml.items()}

print("Data loaded successfully.")
for lbl in SEGMENTS:
    print(f"  {lbl}: ML fire={len(fire_ml[lbl])} rows, HV fire={len(fire_hv[lbl])} rows")


Data loaded successfully.
  I-210 EB: ML fire=4715 rows, HV fire=1435 rows
  I-210 WB: ML fire=5535 rows, HV fire=1640 rows
  SR-134 WB: ML fire=2665 rows, HV fire=2460 rows


## Cell 2 — Macro: Full Day Corridor Overview (06:00–23:00)

4 lines per corridor subplot: Baseline Flow, Fire Day Flow, Baseline Speed, Fire Day Speed.  
Dual y-axis (flow left, speed right). Vertical dashed marker at fire ignition (18:18).

In [2]:
from IPython.display import display, HTML

POST_START, POST_END = "18:00:00", "23:00:00"

def _post(df):
    return df[(df["Time"] >= POST_START) & (df["Time"] <= POST_END)]

def _delta(fire, base):
    if base == 0:
        return "—"
    d = (fire - base) / base * 100
    sign = "▲" if d > 0 else "▼"
    return f"{sign} {abs(d):.1f}%"

def _color(val):
    if not isinstance(val, str) or val == "—":
        return ""
    if "▼" in val:
        return "color: #d62728; font-weight: bold"
    if "▲" in val:
        return "color: #2ca02c; font-weight: bold"
    return ""

TABLE_STYLES = [
    {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"),
                                  ("padding", "8px 12px"), ("text-align", "center")]},
    {"selector": "td", "props": [("padding", "7px 12px"), ("border-bottom", "1px solid #ddd")]},
    {"selector": "tr:nth-child(even)", "props": [("background-color", "#f8f9fa")]},
]

# ══════════════════════════════════════════════════════════════════════════
# MACRO SUMMARY  — Corridor-level totals (post-ignition 18:00–23:00)
# ══════════════════════════════════════════════════════════════════════════
macro_rows = []
for lbl in SEGMENTS:
    for lane_type, fd_dict, bd_dict in [
        ("ML (Mixed)",  fire_ml,  baseline_ml),
        ("HOV",         fire_hv,  baseline_hv),
    ]:
        fd = _post(fd_dict[lbl])
        bd = _post(bd_dict[lbl])
        if fd.empty:
            continue

        fi_flow  = fd["Total Flow (veh/5min)"].sum()
        bl_flow  = bd["Total Flow (veh/5min)"].sum()
        fi_speed = fd["Avg Speed (mph)"].mean()
        bl_speed = bd["Avg Speed (mph)"].mean()

        macro_rows.append({
            "Corridor":                  lbl,
            "Lane Type":                 lane_type,
            "Baseline Total Flow":       f"{bl_flow:,.0f}",
            "Fire Day Total Flow":       f"{fi_flow:,.0f}",
            "Flow Δ":                    _delta(fi_flow, bl_flow),
            "Baseline Avg Speed (mph)":  f"{bl_speed:.1f}",
            "Fire Day Avg Speed (mph)":  f"{fi_speed:.1f}",
            "Speed Δ":                   _delta(fi_speed, bl_speed),
        })

macro_df = pd.DataFrame(macro_rows)

display(HTML("<h3>📊 Macro Summary — Post-Ignition (18:00–23:00)</h3>"
             "<p><em>Total flow = sum across all stations × all 5-min intervals in window. "
             "Avg speed = mean across all stations × intervals.</em></p>"))
display(
    macro_df.style
        .set_properties(**{"text-align": "center"})
        .set_properties(subset=["Corridor", "Lane Type"], **{"text-align": "left", "font-weight": "bold"})
        .map(_color, subset=["Flow Δ", "Speed Δ"])
        .set_table_styles(TABLE_STYLES)
        .hide(axis="index")
)

# ══════════════════════════════════════════════════════════════════════════
# MICRO SUMMARY  — Per-lane breakdown (post-ignition 18:00–23:00)
# ══════════════════════════════════════════════════════════════════════════
micro_rows = []
for lbl in SEGMENTS:
    fd_ml_p = _post(fire_ml[lbl])
    bd_ml_p = _post(baseline_ml[lbl])
    fd_hv_p = _post(fire_hv[lbl])
    bd_hv_p = _post(baseline_hv[lbl])

    # ML lanes
    for n in range(1, 7):
        fc = f"Lane{n} Flow (veh/5min)"
        sc = f"Lane{n} Avg Speed (mph)"
        if fc not in fd_ml_p.columns or fd_ml_p[fc].isna().all():
            continue

        fi_flow  = fd_ml_p[fc].sum()
        bl_flow  = bd_ml_p[fc].sum()
        fi_speed = fd_ml_p[sc].mean()
        bl_speed = bd_ml_p[sc].mean()

        micro_rows.append({
            "Corridor":                  lbl,
            "Lane":                      f"Lane {n} (ML)",
            "Baseline Total Flow":       f"{bl_flow:,.0f}",
            "Fire Day Total Flow":       f"{fi_flow:,.0f}",
            "Flow Δ":                    _delta(fi_flow, bl_flow),
            "Baseline Avg Speed (mph)":  f"{bl_speed:.1f}",
            "Fire Day Avg Speed (mph)":  f"{fi_speed:.1f}",
            "Speed Δ":                   _delta(fi_speed, bl_speed),
        })

    # HOV lane
    if not fd_hv_p.empty and "Lane1 Flow (veh/5min)" in fd_hv_p.columns:
        fi_flow  = fd_hv_p["Lane1 Flow (veh/5min)"].sum()
        bl_flow  = bd_hv_p["Lane1 Flow (veh/5min)"].sum()
        fi_speed = fd_hv_p["Lane1 Avg Speed (mph)"].mean()
        bl_speed = bd_hv_p["Lane1 Avg Speed (mph)"].mean()

        micro_rows.append({
            "Corridor":                  lbl,
            "Lane":                      "HOV Lane",
            "Baseline Total Flow":       f"{bl_flow:,.0f}",
            "Fire Day Total Flow":       f"{fi_flow:,.0f}",
            "Flow Δ":                    _delta(fi_flow, bl_flow),
            "Baseline Avg Speed (mph)":  f"{bl_speed:.1f}",
            "Fire Day Avg Speed (mph)":  f"{fi_speed:.1f}",
            "Speed Δ":                   _delta(fi_speed, bl_speed),
        })

micro_df = pd.DataFrame(micro_rows)

display(HTML("<br><h3>🔬 Micro Summary — Per-Lane Breakdown, Post-Ignition (18:00–23:00)</h3>"
             "<p><em>▲ = increased vs baseline &nbsp;|&nbsp; ▼ = decreased vs baseline.<br>"
             "Speed Δ ▼ = congestion &nbsp;|&nbsp; "
             "SR-134 WB Flow Δ ▲ = evacuation surge onto westbound route toward Pasadena.</em></p>"))
display(
    micro_df.style
        .set_properties(**{"text-align": "center"})
        .set_properties(subset=["Corridor", "Lane"], **{"text-align": "left", "font-weight": "bold"})
        .map(_color, subset=["Flow Δ", "Speed Δ"])
        .set_table_styles(TABLE_STYLES)
        .hide(axis="index")
)


Corridor,Lane Type,Baseline Total Flow,Fire Day Total Flow,Flow Δ,Baseline Avg Speed (mph),Fire Day Avg Speed (mph),Speed Δ
I-210 EB,ML (Mixed),"305,351","258,396",▼ 15.4%,62.8,56.0,▼ 10.7%
I-210 EB,HOV,"30,981","26,017",▼ 16.0%,58.9,59.3,▲ 0.7%
I-210 WB,ML (Mixed),"377,995","353,657",▼ 6.4%,62.0,51.4,▼ 17.0%
I-210 WB,HOV,"28,711","24,815",▼ 13.6%,61.5,62.6,▲ 1.8%
SR-134 WB,ML (Mixed),"172,713","200,600",▲ 16.1%,64.5,56.0,▼ 13.1%
SR-134 WB,HOV,"59,376","59,376",▼ 0.0%,64.5,64.5,▼ 0.0%


Corridor,Lane,Baseline Total Flow,Fire Day Total Flow,Flow Δ,Baseline Avg Speed (mph),Fire Day Avg Speed (mph),Speed Δ
I-210 EB,Lane 1 (ML),"69,964","56,064",▼ 19.9%,67.5,63.4,▼ 6.1%
I-210 EB,Lane 2 (ML),"73,985","61,182",▼ 17.3%,65.1,61.6,▼ 5.3%
I-210 EB,Lane 3 (ML),"65,898","56,115",▼ 14.8%,60.9,52.6,▼ 13.6%
I-210 EB,Lane 4 (ML),"61,862","53,949",▼ 12.8%,58.3,49.3,▼ 15.5%
I-210 EB,Lane 5 (ML),"29,708","27,152",▼ 8.6%,54.7,43.6,▼ 20.3%
I-210 EB,Lane 6 (ML),"3,934","3,934",▼ 0.0%,55.8,55.8,▼ 0.0%
I-210 EB,HOV Lane,"30,981","26,017",▼ 16.0%,58.9,59.3,▲ 0.7%
I-210 WB,Lane 1 (ML),"88,775","83,444",▼ 6.0%,68.0,59.8,▼ 12.1%
I-210 WB,Lane 2 (ML),"94,052","90,602",▼ 3.7%,65.3,55.4,▼ 15.3%
I-210 WB,Lane 3 (ML),"78,570","74,957",▼ 4.6%,59.8,47.4,▼ 20.7%


In [3]:
COLORS = {
    "baseline_flow":  "#4393c3",
    "fire_flow":      "#d6604d",
    "baseline_speed": "#74c476",
    "fire_speed":     "#fd8d3c",
}

def make_macro_plot(time_filter=None, title_suffix="Full Day (06:00–23:00)"):
    labels = list(SEGMENTS.keys())
    subplot_titles = []
    for lbl in labels:
        subplot_titles += [f"{lbl} — Flow", f"{lbl} — Speed"]

    fig = make_subplots(
        rows=3, cols=2,
        shared_xaxes=True,
        subplot_titles=subplot_titles,
        vertical_spacing=0.10,
        horizontal_spacing=0.10,
    )

    for row, lbl in enumerate(labels, start=1):
        fa = fire_agg[lbl].copy()
        ba = baseline_agg[lbl].copy()

        if time_filter:
            fa = fa[(fa["Time"] >= time_filter[0]) & (fa["Time"] <= time_filter[1])]
            ba = ba[(ba["Time"] >= time_filter[0]) & (ba["Time"] <= time_filter[1])]

        showlegend = (row == 1)

        # ── Flow (left column) ────────────────────────────────────────────
        fig.add_trace(go.Scatter(x=ba["Time"], y=ba["Total_Flow"],
            name="Baseline Flow", line=dict(color=COLORS["baseline_flow"], dash="dash", width=1.8),
            legendgroup="baseline_flow", showlegend=showlegend),
            row=row, col=1)

        fig.add_trace(go.Scatter(x=fa["Time"], y=fa["Total_Flow"],
            name="Fire Day Flow", line=dict(color=COLORS["fire_flow"], width=2),
            legendgroup="fire_flow", showlegend=showlegend),
            row=row, col=1)

        # ── Speed (right column) ──────────────────────────────────────────
        fig.add_trace(go.Scatter(x=ba["Time"], y=ba["Avg_Speed"],
            name="Baseline Speed", line=dict(color=COLORS["baseline_speed"], dash="dot", width=1.8),
            legendgroup="baseline_speed", showlegend=showlegend),
            row=row, col=2)

        fig.add_trace(go.Scatter(x=fa["Time"], y=fa["Avg_Speed"],
            name="Fire Day Speed", line=dict(color=COLORS["fire_speed"], dash="dot", width=2),
            legendgroup="fire_speed", showlegend=showlegend),
            row=row, col=2)

        # ── Ignition line on both columns ─────────────────────────────────
        for col in [1, 2]:
            fig.add_shape(type="line",
                x0=FIRE_IGNITION, x1=FIRE_IGNITION, y0=0, y1=1,
                xref="x", yref="y domain",
                line=dict(color="crimson", dash="dash", width=1.5),
                row=row, col=col)
        if row == 1:
            fig.add_annotation(x=FIRE_IGNITION, y=1, xref="x", yref="y domain",
                text="Ignition 18:18", showarrow=False,
                xanchor="left", font=dict(color="crimson", size=10),
                row=1, col=1)

        fig.update_yaxes(title_text="Total Flow (veh/5min)", row=row, col=1)
        fig.update_yaxes(title_text="Avg Speed (mph)", row=row, col=2)

    fig.update_xaxes(title_text="Time of Day", row=3, col=1)
    fig.update_xaxes(title_text="Time of Day", row=3, col=2)
    fig.update_layout(
        height=950,
        title=f"Eaton Fire Corridor — {title_suffix}",
        legend=dict(orientation="h", y=-0.06),
        hovermode="x unified",
    )
    fig.show()

make_macro_plot(title_suffix="Full Day (06:00–23:00)")


## Cell 3 — Macro: Post-Ignition Zoom (18:00–23:00)

Same 4 lines per corridor, filtered to the evacuation window only.

In [4]:
make_macro_plot(time_filter=["18:00:00", "23:00:00"], title_suffix="Post-Ignition (18:00–23:00)")

## Cell 4 — Micro: Per-Station Full Day (06:00–23:00)

One line per station per corridor, colored by `Abs_PM` (milepost — Viridis scale: low PM = purple, high PM = yellow).  
Baseline shown as light grey. Flow only (speed omitted to avoid overcrowding).  
Hover to identify individual stations.

In [5]:
LANE_COLORS = {
    "flow":  {"baseline": "#4393c3", "fire": "#d6604d"},
    "speed": {"baseline": "#74c476", "fire": "#fd8d3c"},
    "hov":   {"baseline": "#9467bd", "fire": "#e377c2"},
}

def _ignition_marker(fig):
    fig.add_shape(type="line",
        x0=FIRE_IGNITION, x1=FIRE_IGNITION, y0=0, y1=1,
        xref="x", yref="y domain",
        line=dict(color="crimson", dash="dash", width=1.5))
    fig.add_annotation(x=FIRE_IGNITION, y=1, xref="x", yref="y domain",
        text="Ignition 18:18", showarrow=False,
        xanchor="left", font=dict(color="crimson", size=10))

def _make_fig(fd_series, bd_series, title, y_label, metric):
    colors = LANE_COLORS[metric]
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=bd_series.index, y=bd_series.values,
        name="Baseline",
        line=dict(color=colors["baseline"], dash="dash", width=1.8),
        hovertemplate=f"Baseline: %{{y:.1f}}<extra></extra>"))
    fig.add_trace(go.Scatter(
        x=fd_series.index, y=fd_series.values,
        name="Fire Day",
        line=dict(color=colors["fire"], width=2),
        hovertemplate=f"Fire Day: %{{y:.1f}}<extra></extra>"))
    _ignition_marker(fig)
    fig.update_layout(
        height=400,
        title=title,
        yaxis_title=y_label,
        xaxis_title="Time of Day",
        legend=dict(orientation="h", y=-0.2),
        hovermode="x unified",
    )
    fig.show()

def make_micro_plot(time_filter=None, title_suffix=""):
    """
    Two figures per (corridor × lane): one for Flow, one for Speed.
    HOV lane is included as a separate set of figures at the end of each corridor.
    """
    def _filter(df):
        if time_filter:
            return df[(df["Time"] >= time_filter[0]) & (df["Time"] <= time_filter[1])]
        return df

    for lbl in SEGMENTS:
        fd_ml = _filter(fire_ml[lbl])
        bd_ml = _filter(baseline_ml[lbl])
        fd_hv = _filter(fire_hv[lbl])
        bd_hv = _filter(baseline_hv[lbl])

        # ── ML lanes (Lane 1 … Lane N) ────────────────────────────────────
        for lane_n in range(1, 7):
            flow_col  = f"Lane{lane_n} Flow (veh/5min)"
            speed_col = f"Lane{lane_n} Avg Speed (mph)"

            if flow_col not in fd_ml.columns or fd_ml[flow_col].isna().all():
                continue

            base = f"{lbl} — Lane {lane_n} (ML) — {title_suffix}"

            _make_fig(
                fd_ml.groupby("Time")[flow_col].sum(),
                bd_ml.groupby("Time")[flow_col].sum(),
                title=f"{base} — Flow", y_label="Flow (veh/5min)", metric="flow")

            _make_fig(
                fd_ml.groupby("Time")[speed_col].mean(),
                bd_ml.groupby("Time")[speed_col].mean(),
                title=f"{base} — Speed", y_label="Speed (mph)", metric="speed")

        # ── HOV lane ─────────────────────────────────────────────────────
        if not fd_hv.empty and "Lane1 Flow (veh/5min)" in fd_hv.columns:
            base_hov = f"{lbl} — HOV Lane — {title_suffix}"

            _make_fig(
                fd_hv.groupby("Time")["Lane1 Flow (veh/5min)"].sum(),
                bd_hv.groupby("Time")["Lane1 Flow (veh/5min)"].sum(),
                title=f"{base_hov} — Flow", y_label="Flow (veh/5min)", metric="hov")

            _make_fig(
                fd_hv.groupby("Time")["Lane1 Avg Speed (mph)"].mean(),
                bd_hv.groupby("Time")["Lane1 Avg Speed (mph)"].mean(),
                title=f"{base_hov} — Speed", y_label="Speed (mph)", metric="hov")


## Cell 5 — Micro: Per-Station Post-Ignition (18:00–23:00)

Spatiotemporal fingerprint of the evacuation.  
Shows where along the corridor the surge/collapse originated and how it propagated over milepost.

In [6]:
POST = dict(time_filter=["18:00:00", "23:00:00"], title_suffix="Post-Ignition (18:00–23:00)")
make_micro_plot(**POST)
